# PPC v8 — Gap-Preserving Point Cloud Prediction Network

## Key fixes over v6 (which fills inter-vertebral gaps):

1. **gap_occ at 96³ with dilation=0** — v6 used 48³ with dilation=1 which FILLS gaps
2. **Differentiable gap penalty** via `F.grid_sample` — v7 used `.long()` indexing which kills gradients
3. **Soft KDE-based Z-density matching** — v7 used hard histogram binning (non-differentiable)
4. **Sliced Wasserstein distance** for global distribution matching
5. **Asymmetric Chamfer** — GT→pred weighted 1.5× to ensure coverage of all GT regions
6. **Occupancy-conditioned query init + refinement** — model learns to avoid empty space
7. **Loss ramping** — gap losses start at epoch 15 so model learns basic shape first

In [1]:
"""
PPC v8 — Gap-Preserving Point Cloud Prediction Network
=======================================================
Key fixes over v6 (which fills inter-vertebral gaps):

1. gap_occ at 96³ with dilation=0 (v6: 48³ dilation=1 → FILLS gaps)
2. Differentiable gap penalty via F.grid_sample (v7 used .long() → kills gradients)
3. Soft KDE-based Z-density matching (v7 used hard bins → non-differentiable)
4. Sliced Wasserstein for global distribution match
5. Asymmetric Chamfer (GT→pred weighted more for full coverage)
6. Loss ramping: gap losses start epoch 15 so model learns shape first
"""
import os, sys, json, time, math, random, shutil, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import vtk

warnings.filterwarnings('ignore', category=FutureWarning)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Total VRAM: {total_gb:.1f} GB')
    fraction = min(50.0 / total_gb, 0.95)
    torch.cuda.set_per_process_memory_fraction(fraction, device=0)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

# ── PATHS ─────────────────────────────────────────────────────────────────────
DATA_ROOT   = Path('/apps/app/see_all_ai/dataset/Lumbar_Filtered_1037')
SPLIT_FILE  = DATA_ROOT / 'dataset_split.json'
PROJECT_DIR = Path('/apps/app/pandu/ppc_network_v8')
CKPT_DIR    = PROJECT_DIR / 'checkpoints'
LOG_DIR     = PROJECT_DIR / 'logs'
RESULTS_DIR = PROJECT_DIR / 'results'
for d in [PROJECT_DIR, CKPT_DIR, LOG_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── MODEL CONFIG ──────────────────────────────────────────────────────────────
IMG_SIZE         = 512
COARSE_VOL_SIZE  = 32
AUX_VOL_SIZE     = 48
GAP_VOL_SIZE     = 96   # NEW: high-res for gap detection (v6 used 48 with dilation)
N_POINTS         = 5120
ENC_CHANNELS     = 192
VOL_CHANNELS     = 128
DEC_CHANNELS     = 128
QUERY_DIM        = 256
N_REFINE_STAGES  = 3
PRETRAINED       = True

# ── TRAINING ──────────────────────────────────────────────────────────────────
BATCH_SIZE       = 2
NUM_WORKERS      = 4
EPOCHS           = 250
LR               = 8e-5       # slightly lower than v6 for stable convergence
WEIGHT_DECAY     = 1e-5
WARMUP_EPOCHS    = 10
USE_AMP          = True
GRAD_CLIP        = 1.0

# ── LOSS WEIGHTS (v8 gap-preserving) ──────────────────────────────────────────
W_CHAMFER   = 1.00   # asymmetric chamfer (GT→pred weighted 1.5×)
W_GAP       = 0.50   # empty-voxel penalty ← THE KEY NEW TERM
W_AXIAL     = 0.40   # soft Z-density matching
W_SW        = 0.25   # sliced Wasserstein
W_REPULSION = 0.02   # anisotropic (Z-favored)
W_AUX_OCC   = 0.05   # coarse occupancy supervision
W_OFFSET    = 0.0005
W_PROJ      = 0.02   # 2D chamfer anchor

# ── EVAL ──────────────────────────────────────────────────────────────────────
MAX_EVAL_SAMPLES = 103

# ── CHECKPOINT ────────────────────────────────────────────────────────────────
CKPT_PATH      = CKPT_DIR / 'latest_checkpoint.pth'
BEST_CKPT_PATH = CKPT_DIR / 'best_checkpoint.pth'
TRAINING_LOG   = LOG_DIR  / 'training_log.json'

print('='*72)
print('CONFIGURATION — PPC v8 (Gap-Preserving)')
print('='*72)
cfg = dict(IMG_SIZE=IMG_SIZE, N_POINTS=N_POINTS, GAP_VOL_SIZE=GAP_VOL_SIZE,
           BATCH_SIZE=BATCH_SIZE, EPOCHS=EPOCHS, LR=LR,
           W_CHAMFER=W_CHAMFER, W_GAP=W_GAP, W_AXIAL=W_AXIAL, W_SW=W_SW,
           W_REPULSION=W_REPULSION, N_REFINE_STAGES=N_REFINE_STAGES)
for k, v in cfg.items():
    print(f'  {k:<20} = {v}')

Device: cuda
GPU: NVIDIA A100-SXM4-80GB
Total VRAM: 85.0 GB
CONFIGURATION — PPC v8 (Gap-Preserving)
  IMG_SIZE             = 512
  N_POINTS             = 5120
  GAP_VOL_SIZE         = 96
  BATCH_SIZE           = 2
  EPOCHS               = 250
  LR                   = 8e-05
  W_CHAMFER            = 1.0
  W_GAP                = 0.5
  W_AXIAL              = 0.4
  W_SW                 = 0.25
  W_REPULSION          = 0.02
  N_REFINE_STAGES      = 3


In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# DATA UTILITIES
# ══════════════════════════════════════════════════════════════════════════════

def load_vtk_points(path):
    reader = vtk.vtkPolyDataReader()
    reader.SetFileName(str(path))
    reader.Update()
    poly = reader.GetOutput()
    n = poly.GetNumberOfPoints()
    return np.array([poly.GetPoint(i) for i in range(n)], dtype=np.float32)


def save_vtk_points(points, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    vp = vtk.vtkPoints()
    for p in points:
        vp.InsertNextPoint(float(p[0]), float(p[1]), float(p[2]))
    vc = vtk.vtkCellArray()
    for i in range(len(points)):
        vc.InsertNextCell(1)
        vc.InsertCellPoint(i)
    poly = vtk.vtkPolyData()
    poly.SetPoints(vp)
    poly.SetVerts(vc)
    w = vtk.vtkPolyDataWriter()
    w.SetFileName(str(output_path))
    w.SetInputData(poly)
    w.SetFileTypeToASCII()
    w.Write()
    return output_path


def load_drr_image(path, size=IMG_SIZE):
    img = Image.open(path).convert('L')
    if img.size != (size, size):
        img = img.resize((size, size), Image.BILINEAR)
    return np.array(img, dtype=np.float32) / 255.0


def load_projection_matrix(path):
    with open(path, 'r') as f:
        vals = [float(v) for v in f.read().split()]
    return np.array(vals, dtype=np.float32).reshape(3, 4)


def load_split(split_path=SPLIT_FILE):
    with open(split_path, 'r') as f:
        d = json.load(f)
    return d['train'], d['val'], d['test']


def append_training_log(log_path, epoch_record):
    log = {'records': []}
    if log_path.exists():
        with open(log_path, 'r') as f:
            log = json.load(f)
    log['records'].append(epoch_record)
    tmp = log_path.with_suffix('.json.tmp')
    with open(tmp, 'w') as f:
        json.dump(log, f, indent=2)
    tmp.replace(log_path)


def points_to_local(points, center, scale):
    return (points - center[None]) / (scale[None] + 1e-6)


def local_to_world(points_local, center, scale):
    return points_local * scale[:, None, :] + center[:, None, :]


def compute_scale(gt_full):
    mins = gt_full.min(axis=0)
    maxs = gt_full.max(axis=0)
    extent = (maxs - mins).astype(np.float32)
    scale = extent * 0.55 + np.array([20., 20., 30.], dtype=np.float32)
    return np.maximum(scale, np.array([50., 50., 80.], dtype=np.float32))


def make_aux_occupancy(points_local, vol_size=AUX_VOL_SIZE, dilate=1):
    """Coarse occupancy for U-Net supervision (dilated = filled)."""
    pts = np.clip((np.asarray(points_local, np.float32) + 1.0) * 0.5, 0.0, 0.999999)
    idx = np.clip(np.floor(pts * vol_size).astype(np.int32), 0, vol_size - 1)
    occ = np.zeros((vol_size, vol_size, vol_size), dtype=np.float32)
    for dx in range(-dilate, dilate + 1):
        for dy in range(-dilate, dilate + 1):
            for dz in range(-dilate, dilate + 1):
                x = np.clip(idx[:, 0] + dx, 0, vol_size - 1)
                y = np.clip(idx[:, 1] + dy, 0, vol_size - 1)
                z = np.clip(idx[:, 2] + dz, 0, vol_size - 1)
                occ[x, y, z] = 1.0
    return occ


def make_gap_occupancy(points_local, vol_size=GAP_VOL_SIZE, dilate=0):
    """HIGH-RES occupancy with NO dilation — gaps stay empty.
    This is THE critical difference from v6.
    At 96³ with typical scale, Z-voxel ≈ 2.5mm → resolves 3-5mm gaps.
    """
    pts = np.clip((np.asarray(points_local, np.float32) + 1.0) * 0.5, 0.0, 0.999999)
    idx = np.clip(np.floor(pts * vol_size).astype(np.int32), 0, vol_size - 1)
    occ = np.zeros((vol_size, vol_size, vol_size), dtype=np.float32)
    occ[idx[:, 0], idx[:, 1], idx[:, 2]] = 1.0
    return occ


def flip_projection_matrix_horizontal(P, img_size=IMG_SIZE):
    F_mat = np.array([
        [-1,  0,  img_size - 1],
        [ 0,  1,  0           ],
        [ 0,  0,  1           ],
    ], dtype=np.float32)
    return F_mat @ P


AUG_INTENSITY = 0.15
AUG_FLIP_PROB = 0.5

In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# DATASET
# ══════════════════════════════════════════════════════════════════════════════

class LumbarDatasetV8(Dataset):
    """v8 dataset: adds gap_occ field (96³, no dilation) for gap penalty loss."""

    def __init__(self, patient_ids, data_root=DATA_ROOT, img_size=IMG_SIZE, augment=False):
        self.patient_ids = patient_ids
        self.data_root   = Path(data_root)
        self.img_size    = img_size
        self.augment     = augment
        self.img_norm    = transforms.Normalize(mean=[0.485], std=[0.229])

    def __len__(self):
        return len(self.patient_ids)

    def __getitem__(self, idx):
        pid  = self.patient_ids[idx]
        pdir = self.data_root / pid

        drr_ap = load_drr_image(pdir / 'AP_0'  / 'drr_AP_0.png',  self.img_size)
        drr_lp = load_drr_image(pdir / 'LP_90' / 'drr_LP_90.png', self.img_size)
        P_ap   = load_projection_matrix(pdir / 'AP_0'  / 'P_AP_0.txt')
        P_lp   = load_projection_matrix(pdir / 'LP_90' / 'P_LP_90.txt')
        gt_full = load_vtk_points(pdir / 'gt_ppc.vtk')

        center = gt_full.mean(axis=0).astype(np.float32)
        scale  = compute_scale(gt_full)
        gt_local_full = np.clip(points_to_local(gt_full, center, scale), -1.0, 1.0)

        # Coarse occupancy for U-Net supervision (dilated)
        aux_occ = make_aux_occupancy(gt_local_full, AUX_VOL_SIZE, dilate=1)
        # High-res occupancy for gap penalty (NOT dilated)
        gap_occ = make_gap_occupancy(gt_local_full, GAP_VOL_SIZE, dilate=0)

        # Subsample GT
        n = len(gt_full)
        sel = np.random.choice(n, N_POINTS, replace=(n < N_POINTS))
        gt_world = gt_full[sel].astype(np.float32)
        gt_local = gt_local_full[sel].astype(np.float32)

        # Augmentation
        if self.augment:
            for drr in [drr_ap, drr_lp]:
                jitter = 1.0 + (np.random.rand() * 2 - 1) * AUG_INTENSITY
                drr[:] = np.clip(drr * jitter, 0.0, 1.0)
            if np.random.rand() < AUG_FLIP_PROB:
                drr_ap = drr_ap[:, ::-1].copy()
                drr_lp = drr_lp[:, ::-1].copy()
                P_ap = flip_projection_matrix_horizontal(P_ap, self.img_size)
                P_lp = flip_projection_matrix_horizontal(P_lp, self.img_size)

        drr_ap_t = self.img_norm(torch.from_numpy(drr_ap).unsqueeze(0).float())
        drr_lp_t = self.img_norm(torch.from_numpy(drr_lp).unsqueeze(0).float())

        return {
            'drr_ap':       drr_ap_t,
            'drr_lp':       drr_lp_t,
            'P_ap':         torch.from_numpy(P_ap).float(),
            'P_lp':         torch.from_numpy(P_lp).float(),
            'gt_ppc_world': torch.from_numpy(gt_world).float(),
            'gt_ppc_local': torch.from_numpy(gt_local).float(),
            'aux_occ':      torch.from_numpy(aux_occ).float(),
            'gap_occ':      torch.from_numpy(gap_occ).float(),  # NEW
            'center':       torch.from_numpy(center).float(),
            'scale':        torch.from_numpy(scale).float(),
            'patient_id':   pid,
        }

In [4]:
train_ids, val_ids, test_ids = load_split()
print(f'Split: train={len(train_ids)}  val={len(val_ids)}  test={len(test_ids)}')

Split: train=829  val=103  test=105


In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# MODEL ARCHITECTURE (encoder + lift + fusion + U-Net same as v6)
# ══════════════════════════════════════════════════════════════════════════════

class Encoder2D(nn.Module):
    def __init__(self, in_channels=1, out_channels=ENC_CHANNELS, pretrained=PRETRAINED):
        super().__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        new_c1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        if pretrained:
            with torch.no_grad():
                new_c1.weight[:] = base.conv1.weight.mean(dim=1, keepdim=True)
        base.conv1 = new_c1
        self.stem   = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.proj   = nn.Conv2d(256, out_channels, kernel_size=1)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return self.proj(x)


class FeatureLift(nn.Module):
    def __init__(self, in_channels=ENC_CHANNELS, out_channels=VOL_CHANNELS, depth=COARSE_VOL_SIZE):
        super().__init__()
        self.depth_embed = nn.Parameter(torch.randn(1, in_channels, depth, 1, 1) * 0.02)
        self.refine = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, 3, padding=1),
            nn.GroupNorm(8, out_channels), nn.GELU(),
            nn.Conv3d(out_channels, out_channels, 3, padding=1),
            nn.GroupNorm(8, out_channels), nn.GELU(),
        )

    def forward(self, feat_2d):
        B, C, H, W = feat_2d.shape
        vol = feat_2d.unsqueeze(2).expand(B, C, COARSE_VOL_SIZE, H, W).contiguous()
        vol = vol + self.depth_embed
        return self.refine(vol)


class BiplanarFusion(nn.Module):
    def __init__(self, in_channels=VOL_CHANNELS, out_channels=VOL_CHANNELS):
        super().__init__()
        self.fuse = nn.Sequential(
            nn.Conv3d(in_channels * 2, out_channels, 1),
            nn.GroupNorm(8, out_channels), nn.GELU(),
            nn.Conv3d(out_channels, out_channels, 3, padding=1),
            nn.GroupNorm(8, out_channels), nn.GELU(),
        )

    def forward(self, ap_vol, lp_vol):
        ap_aligned = ap_vol.permute(0, 1, 4, 2, 3).contiguous()
        lp_aligned = lp_vol.permute(0, 1, 2, 4, 3).contiguous()
        return self.fuse(torch.cat([ap_aligned, lp_aligned], dim=1))


class Block3D(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_c, out_c, 3, padding=1), nn.GroupNorm(8, out_c), nn.GELU(),
            nn.Conv3d(out_c, out_c, 3, padding=1), nn.GroupNorm(8, out_c), nn.GELU(),
        )
    def forward(self, x): return self.block(x)


class CoarseUNet3D(nn.Module):
    def __init__(self, in_channels=VOL_CHANNELS, feat_channels=DEC_CHANNELS):
        super().__init__()
        self.enc1       = Block3D(in_channels, 96)
        self.down1      = nn.Conv3d(96,  160, 2, stride=2)
        self.enc2       = Block3D(160, 160)
        self.down2      = nn.Conv3d(160, 224, 2, stride=2)
        self.bottleneck = Block3D(224, 224)
        self.up2        = nn.ConvTranspose3d(224, 160, 2, stride=2)
        self.dec2       = Block3D(320, 160)
        self.up1        = nn.ConvTranspose3d(160, 96,  2, stride=2)
        self.dec1       = Block3D(192, feat_channels)
        self.aux_head   = nn.Sequential(
            nn.Conv3d(feat_channels, feat_channels // 2, 3, padding=1),
            nn.GELU(),
            nn.Conv3d(feat_channels // 2, 1, 1),
        )

    def forward(self, x):
        e1  = self.enc1(x)
        e2  = self.enc2(self.down1(e1))
        b   = self.bottleneck(self.down2(e2))
        d2  = self.dec2(torch.cat([self.up2(b), e2], dim=1))
        d1  = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        aux = F.interpolate(self.aux_head(d1),
                            size=(AUX_VOL_SIZE,) * 3,
                            mode='trilinear', align_corners=True)
        return d1, aux


# ── QUERY INITIALIZER v8: occupancy-conditioned ──────────────────────────────
class QueryInitializerV8(nn.Module):
    """Uses predicted occupancy to gate point offsets.
    Points in predicted-empty regions get smaller offsets → stay near grid
    → gap_penalty_loss pushes them to occupied regions.
    """
    def __init__(self, n_points=N_POINTS, feat_channels=DEC_CHANNELS):
        super().__init__()
        xs = np.linspace(-0.8, 0.8, 16)
        ys = np.linspace(-0.8, 0.8, 16)
        zs = np.linspace(-0.9, 0.9, 20)
        grid = np.stack(np.meshgrid(xs, ys, zs, indexing='ij'), -1).reshape(-1, 3).astype(np.float32)
        self.register_buffer('base_queries', torch.from_numpy(grid))
        self.n_points = n_points
        self.global_head = nn.Sequential(
            nn.AdaptiveAvgPool3d(1), nn.Flatten(),
            nn.Linear(feat_channels, 256), nn.GELU(),
            nn.Linear(256, n_points * 3),
        )

    def forward(self, vol_feat, aux_occ_logits=None):
        B = vol_feat.shape[0]
        offset = self.global_head(vol_feat).view(B, self.n_points, 3)
        raw_q = self.base_queries[None] + 0.20 * torch.tanh(offset)

        if aux_occ_logits is not None:
            # Sample predicted occupancy at query positions
            gs = torch.stack([raw_q[..., 2], raw_q[..., 1], raw_q[..., 0]], dim=-1)
            gs = gs.view(B, self.n_points, 1, 1, 3)
            occ_vals = F.grid_sample(
                aux_occ_logits, gs, mode='bilinear',
                align_corners=True, padding_mode='border'
            ).view(B, self.n_points)
            gate = torch.sigmoid(occ_vals).unsqueeze(-1)  # 0=empty, 1=bone
            # In empty regions, reduce offset → point stays near grid center
            # gap_penalty_loss will then push it toward occupied space
            raw_q = self.base_queries[None] + gate * 0.25 * torch.tanh(offset)

        return torch.clamp(raw_q, -1.0, 1.0)


# ── PROJECTION HELPERS ────────────────────────────────────────────────────────
def project_points(P, points_world, img_size=IMG_SIZE):
    B, N, _ = points_world.shape
    ones  = torch.ones(B, N, 1, device=points_world.device, dtype=points_world.dtype)
    homog = torch.cat([points_world, ones], dim=-1)
    uvw   = torch.bmm(homog, P.transpose(1, 2))
    z     = uvw[..., 2:3].clamp(min=1e-6)
    uv    = uvw[..., :2] / z
    uv_norm = (uv / (img_size - 1.0)) * 2.0 - 1.0
    return uv, uv_norm, torch.log(z)


def sample_2d_features(feat_map, uv_norm):
    grid = uv_norm.view(feat_map.shape[0], -1, 1, 2)
    samp = F.grid_sample(feat_map, grid, mode='bilinear',
                         align_corners=True, padding_mode='border')
    return samp.squeeze(-1).transpose(1, 2)


def sample_3d_features(vol_feat, points_local):
    grid = torch.stack([points_local[..., 2],
                        points_local[..., 1],
                        points_local[..., 0]], dim=-1)
    grid = grid.view(points_local.shape[0], -1, 1, 1, 3)
    samp = F.grid_sample(vol_feat, grid, mode='bilinear',
                         align_corners=True, padding_mode='border')
    return samp.squeeze(-1).squeeze(-1).transpose(1, 2)


# ── REFINEMENT with occupancy signal ─────────────────────────────────────────
class RefinementStageV8(nn.Module):
    """v8: MLP also receives sampled occupancy → learns to avoid empty regions."""
    def __init__(self, feat_2d=ENC_CHANNELS, feat_3d=DEC_CHANNELS, hidden=QUERY_DIM):
        super().__init__()
        # +1 for occupancy signal compared to v6
        in_dim = feat_2d * 2 + feat_3d + 3 + 2 + 2 + 1
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU(),
            nn.Linear(hidden, 3),
        )

    def forward(self, q_local, vol_feat, feat_ap, feat_lp, P_ap, P_lp,
                center, scale, aux_occ_logits=None):
        q_world = local_to_world(q_local, center, scale)
        _, uvn_ap, _ = project_points(P_ap, q_world)
        _, uvn_lp, _ = project_points(P_lp, q_world)
        f3d = sample_3d_features(vol_feat, q_local)
        fap = sample_2d_features(feat_ap, uvn_ap)
        flp = sample_2d_features(feat_lp, uvn_lp)

        # Sample occupancy at current query positions
        B, N, _ = q_local.shape
        if aux_occ_logits is not None:
            gs = torch.stack([q_local[..., 2], q_local[..., 1], q_local[..., 0]], dim=-1)
            gs = gs.view(B, N, 1, 1, 3)
            occ_val = F.grid_sample(
                aux_occ_logits, gs, mode='bilinear',
                align_corners=True, padding_mode='border'
            ).view(B, N, 1)
        else:
            occ_val = torch.zeros(B, N, 1, device=q_local.device)

        x = torch.cat([f3d, fap, flp, q_local, uvn_ap, uvn_lp, occ_val], dim=-1)
        delta = 0.25 * torch.tanh(self.mlp(x))
        return q_local + delta, {'delta': delta}


# ── FULL NETWORK ──────────────────────────────────────────────────────────────
class PPCNetV8(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder_ap = Encoder2D()
        self.encoder_lp = Encoder2D()
        self.lift_ap    = FeatureLift()
        self.lift_lp    = FeatureLift()
        self.fusion     = BiplanarFusion()
        self.coarse3d   = CoarseUNet3D()
        self.query_init = QueryInitializerV8()
        self.stages     = nn.ModuleList([RefinementStageV8() for _ in range(N_REFINE_STAGES)])

    def forward(self, drr_ap, drr_lp, P_ap, P_lp, center, scale):
        feat_ap = self.encoder_ap(drr_ap)
        feat_lp = self.encoder_lp(drr_lp)
        vol_ap  = self.lift_ap(feat_ap)
        vol_lp  = self.lift_lp(feat_lp)
        fused            = self.fusion(vol_ap, vol_lp)
        vol_feat, aux_occ = self.coarse3d(fused)

        q_local = self.query_init(vol_feat, aux_occ)

        stage_aux = []
        for stage in self.stages:
            q_local, aux = stage(q_local, vol_feat, feat_ap, feat_lp,
                                 P_ap, P_lp, center, scale, aux_occ)
            stage_aux.append(aux)

        q_local = torch.clamp(q_local, -1.0, 1.0)
        q_world = local_to_world(q_local, center, scale)

        return {
            'pred_local':     q_local,
            'pred_world':     q_world,
            'aux_occ_logits': aux_occ,
            'stage_aux':      stage_aux,
        }

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

_test_model = PPCNetV8()
print(f'PPCNetV8 parameters: {count_params(_test_model)/1e6:.1f} M')
del _test_model

PPCNetV8 parameters: 21.8 M


In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# LOSS FUNCTIONS — v8 gap-preserving (ALL differentiable)
# ══════════════════════════════════════════════════════════════════════════════

def pairwise_sqdist(x, y):
    x2 = (x ** 2).sum(-1, keepdim=True)
    y2 = (y ** 2).sum(-1).unsqueeze(1)
    xy = torch.bmm(x, y.transpose(1, 2))
    return (x2 + y2 - 2 * xy).clamp_min(0.0)


def asymmetric_chamfer_loss(pred, gt, w_p2g=1.0, w_g2p=1.5, trunc_mm=15.0):
    """GT→pred weighted MORE so model covers all GT regions including gaps.
    Pred→GT truncated to avoid pulling gap-filling pts toward bone.
    """
    d2 = pairwise_sqdist(pred, gt)
    d_p2g = torch.clamp(d2.min(dim=2)[0], max=trunc_mm**2)
    d_g2p = d2.min(dim=1)[0]
    return w_p2g * d_p2g.mean() + w_g2p * d_g2p.mean()


def gap_penalty_loss(pred_local, gap_occ_gt):
    """Penalize points in empty GT voxels via F.grid_sample (differentiable).
    gap_occ_gt: (B, D, H, W) binary, 1=bone 0=gap.
    """
    B, N, _ = pred_local.shape
    occ = gap_occ_gt.unsqueeze(1)  # (B,1,D,H,W)
    grid = torch.stack([pred_local[..., 2], pred_local[..., 1],
                        pred_local[..., 0]], dim=-1)
    grid = grid.view(B, N, 1, 1, 3)
    sampled = F.grid_sample(occ, grid, mode='bilinear',
                            align_corners=True, padding_mode='border')
    occ_at_pts = sampled.view(B, N)
    return (1.0 - occ_at_pts).clamp(min=0).mean()


def soft_axial_density_loss(pred_local, gt_local, n_bins=60, sigma=0.025):
    """Match Z-axis density via Gaussian KDE (fully differentiable).
    Gaps appear as valleys; this loss enforces matching valleys.
    """
    pred_z = pred_local[..., 2]
    gt_z   = gt_local[..., 2]
    centers = torch.linspace(-1.0, 1.0, n_bins, device=pred_local.device)
    p_kde = torch.exp(-0.5 * ((pred_z.unsqueeze(-1) - centers) / sigma)**2)
    g_kde = torch.exp(-0.5 * ((gt_z.unsqueeze(-1) - centers) / sigma)**2)
    p_d = p_kde.sum(dim=1)
    g_d = g_kde.sum(dim=1)
    p_d = p_d / (p_d.sum(dim=1, keepdim=True) + 1e-8)
    g_d = g_d / (g_d.sum(dim=1, keepdim=True) + 1e-8)
    return (p_d - g_d).abs().sum(dim=1).mean()


def sliced_wasserstein_loss(pred, gt, n_proj=50):
    """1D Wasserstein along random projections — captures distributional gaps."""
    dirs = F.normalize(torch.randn(n_proj, 3, device=pred.device), dim=-1)
    p_proj = torch.einsum('bnd,pd->bnp', pred, dirs)
    g_proj = torch.einsum('bnd,pd->bnp', gt, dirs)
    return ((p_proj.sort(dim=1)[0] - g_proj.sort(dim=1)[0])**2).mean()


def repulsion_loss_aniso(points, k=8, h_xy=0.020, h_z=0.040):
    """Anisotropic repulsion: larger Z-tolerance preserves vertical gaps."""
    d2 = pairwise_sqdist(points, points)
    B, N, _ = d2.shape
    eye = torch.eye(N, device=points.device, dtype=torch.bool).unsqueeze(0)
    d2 = d2.masked_fill(eye, float('inf'))
    knn_idx = torch.topk(d2, k=k, dim=-1, largest=False).indices
    pts_exp = points.unsqueeze(2).expand(B, N, k, 3)
    nbr = torch.gather(points.unsqueeze(1).expand(B, N, N, 3), 2,
                        knn_idx.unsqueeze(-1).expand(B, N, k, 3))
    diff = pts_exp - nbr
    d2_a = (diff[..., 0]**2 + diff[..., 1]**2) / (h_xy**2) + diff[..., 2]**2 / (h_z**2)
    return torch.exp(-d2_a).mean()


def project_consistency_loss(pred_world, gt_world, P_ap, P_lp):
    uvp_ap, _, _ = project_points(P_ap, pred_world)
    uvg_ap, _, _ = project_points(P_ap, gt_world)
    uvp_lp, _, _ = project_points(P_lp, pred_world)
    uvg_lp, _, _ = project_points(P_lp, gt_world)
    def ch(a, b):
        d2 = pairwise_sqdist(a, b)
        return 0.5 * (d2.min(2)[0].mean() + d2.min(1)[0].mean())
    return ch(uvp_ap, uvg_ap) + ch(uvp_lp, uvg_lp)


def dice_loss_from_logits(logits, targets, eps=1e-6):
    probs   = torch.sigmoid(logits).reshape(logits.shape[0], -1)
    targets = targets.reshape(targets.shape[0], -1)
    inter   = (probs * targets).sum(dim=1)
    denom   = probs.sum(dim=1) + targets.sum(dim=1)
    return 1.0 - ((2 * inter + eps) / (denom + eps)).mean()


def total_loss(output, batch, current_epoch=0):
    """Full v8 loss with gap-preservation terms.
    Schedule: gap/axial/SW ramp from epoch 15→40.
    """
    pred_w = output['pred_world']
    pred_l = output['pred_local']
    gt_w   = batch['gt_ppc_world']
    gt_l   = batch['gt_ppc_local']

    l_ch  = asymmetric_chamfer_loss(pred_w, gt_w)
    l_gap = gap_penalty_loss(pred_l, batch['gap_occ'])
    l_ax  = soft_axial_density_loss(pred_l, gt_l)
    l_sw  = sliced_wasserstein_loss(pred_l, gt_l)
    l_rep = repulsion_loss_aniso(pred_l)
    l_aux = (F.binary_cross_entropy_with_logits(
                output['aux_occ_logits'].squeeze(1), batch['aux_occ'])
             + dice_loss_from_logits(
                output['aux_occ_logits'].squeeze(1), batch['aux_occ']))
    l_off = torch.stack([a['delta'].abs().mean() for a in output['stage_aux']]).mean()

    w_proj_eff = W_PROJ + max(0.0, (30 - current_epoch) / 30.0) * 0.08
    l_proj = project_consistency_loss(pred_w, gt_w, batch['P_ap'], batch['P_lp'])

    # Ramp: gap losses 0→1 over epochs 15→40
    ramp = min(1.0, max(0.0, (current_epoch - 15) / 25.0))

    loss = (W_CHAMFER   * l_ch
          + W_GAP       * ramp * l_gap
          + W_AXIAL     * ramp * l_ax
          + W_SW        * ramp * l_sw
          + W_REPULSION * l_rep
          + W_AUX_OCC   * l_aux
          + W_OFFSET    * l_off
          + w_proj_eff  * l_proj)

    bd = {
        'total': float(loss.detach().cpu()),
        'chamfer': float(l_ch.detach().cpu()),
        'gap': float(l_gap.detach().cpu()),
        'axial': float(l_ax.detach().cpu()),
        'sw': float(l_sw.detach().cpu()),
        'repulsion': float(l_rep.detach().cpu()),
        'aux_occ': float(l_aux.detach().cpu()),
        'offset': float(l_off.detach().cpu()),
        'proj': float(l_proj.detach().cpu()),
        'w_proj_eff': float(w_proj_eff),
        'ramp': float(ramp),
    }
    return loss, bd

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# TRAINING UTILITIES
# ══════════════════════════════════════════════════════════════════════════════

def save_checkpoint(path, model, optimizer, scheduler, scaler, epoch, best_val, history):
    state = {
        'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict() if scheduler else None,
        'scaler': scaler.state_dict() if scaler else None,
        'epoch': epoch, 'best_val_loss': best_val,
        'training_history': history,
        'config': {'version': 'v8_gap_preserving'},
        'timestamp': datetime.utcnow().isoformat(),
    }
    tmp = path.with_suffix('.pth.tmp')
    torch.save(state, tmp)
    tmp.replace(path)


def load_checkpoint(path, model, optimizer=None, scheduler=None, scaler=None):
    state = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(state['model'])
    if optimizer and state.get('optimizer'): optimizer.load_state_dict(state['optimizer'])
    if scheduler and state.get('scheduler'): scheduler.load_state_dict(state['scheduler'])
    if scaler    and state.get('scaler'):    scaler.load_state_dict(state['scaler'])
    start = state['epoch'] + 1
    best  = state.get('best_val_loss', float('inf'))
    hist  = state.get('training_history', [])
    print(f'  Resumed from epoch {start} | best_val_loss: {best:.4f}')
    return start, best, hist


def collate_keep_strings(batch):
    out = {}
    for k in batch[0]:
        vals = [b[k] for b in batch]
        out[k] = torch.stack(vals, 0) if isinstance(vals[0], torch.Tensor) else vals
    return out


def move_to_device(batch):
    return {k: (v.to(device, non_blocking=True) if isinstance(v, torch.Tensor) else v)
            for k, v in batch.items()}


def chamfer_metrics_np(pred, gt):
    pred = np.asarray(pred, np.float32)
    gt   = np.asarray(gt, np.float32)
    d = np.linalg.norm(pred[:, None] - gt[None], axis=-1)
    d_pg = d.min(axis=1)
    d_gp = d.min(axis=0)
    def fs(th):
        p = (d_pg <= th).mean(); r = (d_gp <= th).mean()
        return 2*p*r/(p+r) if (p+r) > 0 else 0.0
    return {
        'chamfer_mm':   float(0.5*(d_pg.mean()+d_gp.mean())),
        'fscore_1mm':   float(fs(1.0)),
        'fscore_2mm':   float(fs(2.0)),
        'fscore_5mm':   float(fs(5.0)),
        'hausdorff_95': float(max(np.percentile(d_pg,95), np.percentile(d_gp,95))),
    }


@torch.no_grad()
def evaluate(model, loader, max_samples=MAX_EVAL_SAMPLES, current_epoch=0):
    model.eval()
    acc = {k: 0.0 for k in ('total','chamfer','gap','axial','sw','repulsion','aux_occ','offset')}
    n_bat = 0; metrics = []; n_done = 0
    for batch in loader:
        batch  = move_to_device(batch)
        output = model(batch['drr_ap'], batch['drr_lp'],
                       batch['P_ap'], batch['P_lp'],
                       batch['center'], batch['scale'])
        _, bd = total_loss(output, batch, current_epoch)
        for k in acc:
            acc[k] += bd.get(k, 0.0)
        n_bat += 1
        if n_done < max_samples:
            pred = output['pred_world'].cpu().numpy()
            gt   = batch['gt_ppc_world'].cpu().numpy()
            for b in range(pred.shape[0]):
                if n_done >= max_samples: break
                metrics.append(chamfer_metrics_np(pred[b], gt[b]))
                n_done += 1
    result = {k: acc[k]/max(1, n_bat) for k in acc}
    if metrics:
        result.update({
            'mean_mm':      float(np.mean([m['chamfer_mm'] for m in metrics])),
            'std_mm':       float(np.std([m['chamfer_mm'] for m in metrics])),
            'fscore_1mm':   float(np.mean([m['fscore_1mm'] for m in metrics])),
            'fscore_2mm':   float(np.mean([m['fscore_2mm'] for m in metrics])),
            'fscore_5mm':   float(np.mean([m['fscore_5mm'] for m in metrics])),
            'hausdorff_95': float(np.mean([m['hausdorff_95'] for m in metrics])),
            'n_patients':   n_done,
        })
    else:
        result.update(dict(mean_mm=float('inf'), std_mm=0.0,
                           fscore_1mm=0.0, fscore_2mm=0.0,
                           fscore_5mm=0.0, hausdorff_95=float('inf'), n_patients=0))
    return result

In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# TRAINING LOOP
# ══════════════════════════════════════════════════════════════════════════════

train_ds = LumbarDatasetV8(train_ids, augment=True)
val_ds   = LumbarDatasetV8(val_ids, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          collate_fn=collate_keep_strings, persistent_workers=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          collate_fn=collate_keep_strings, persistent_workers=True)

print(f'Train: {len(train_ds)} patients → {len(train_loader)} batches/epoch')
print(f'Val  : {len(val_ds)}   patients → {len(val_loader)}   batches/epoch')

model     = PPCNetV8().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler    = torch.cuda.amp.GradScaler(enabled=USE_AMP and device.type == 'cuda')

warmup_steps = max(1, WARMUP_EPOCHS * len(train_loader))
total_steps  = max(1, EPOCHS * len(train_loader))

def lr_lambda(step):
    if step < warmup_steps:
        return float(step + 1) / float(warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

start_epoch   = 0
best_val_loss = float('inf')
best_chamfer  = float('inf')
history       = []

if CKPT_PATH.exists():
    print('Resuming from checkpoint …')
    start_epoch, best_val_loss, history = load_checkpoint(
        CKPT_PATH, model, optimizer, scheduler, scaler)
    best_chamfer = min((r['val'].get('mean_mm', float('inf')) for r in history), default=float('inf'))
else:
    print('No checkpoint found — starting fresh.')

print(f'Model params : {count_params(model)/1e6:.1f} M')
print(f'Start epoch  : {start_epoch + 1}')

for epoch in range(start_epoch, EPOCHS):
    model.train()
    acc = {k: 0.0 for k in ('total','chamfer','gap','axial','sw','repulsion','aux_occ','offset','proj')}

    for bi, batch in enumerate(train_loader, start=1):
        batch = move_to_device(batch)
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP and device.type == 'cuda'):
            output = model(batch['drr_ap'], batch['drr_lp'],
                           batch['P_ap'], batch['P_lp'],
                           batch['center'], batch['scale'])
            loss, bd = total_loss(output, batch, current_epoch=epoch)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        for k in acc:
            acc[k] += bd.get(k, 0.0)

        if bi % 100 == 0 or bi == len(train_loader):
            lr_now = optimizer.param_groups[0]['lr']
            print(f"  [Ep {epoch+1:3d}/{EPOCHS}] {bi:3d}/{len(train_loader)} "
                  f"loss={bd['total']:.4f} ch={bd['chamfer']:.4f} "
                  f"gap={bd['gap']:.4f} ax={bd['axial']:.4f} "
                  f"ramp={bd['ramp']:.2f} lr={lr_now:.2e}")

    train_stats = {k: acc[k]/max(1, len(train_loader)) for k in acc}
    val_stats   = evaluate(model, val_loader, current_epoch=epoch)

    rec = {'epoch': epoch+1, 'train': train_stats, 'val': val_stats,
           'lr': optimizer.param_groups[0]['lr']}
    history.append(rec)
    append_training_log(TRAINING_LOG, rec)

    save_checkpoint(CKPT_PATH, model, optimizer, scheduler, scaler,
                    epoch, best_val_loss, history)

    print(f"[Epoch {epoch+1:3d}/{EPOCHS}] "
          f"train={train_stats['total']:.4f} val={val_stats['total']:.4f}")
    print(f"  Chamfer={val_stats['mean_mm']:.3f}±{val_stats['std_mm']:.3f} mm "
          f"F@1={val_stats['fscore_1mm']:.3f} F@2={val_stats['fscore_2mm']:.3f} "
          f"F@5={val_stats['fscore_5mm']:.3f} HD95={val_stats['hausdorff_95']:.3f}")

    if val_stats['total'] < best_val_loss:
        best_val_loss = val_stats['total']
        save_checkpoint(BEST_CKPT_PATH, model, optimizer, scheduler, scaler,
                        epoch, best_val_loss, history)
        print(f"  ✓ New best val_loss: {best_val_loss:.4f}")

    if val_stats['mean_mm'] < best_chamfer:
        best_chamfer = val_stats['mean_mm']
        save_checkpoint(CKPT_DIR / 'best_chamfer_checkpoint.pth',
                        model, optimizer, scheduler, scaler,
                        epoch, best_val_loss, history)
        print(f"  ✓ New best Chamfer: {best_chamfer:.3f} mm")

print(f'\nTraining complete. Best Chamfer: {best_chamfer:.3f} mm')

Train: 829 patients → 415 batches/epoch
Val  : 103   patients → 52   batches/epoch
No checkpoint found — starting fresh.
Model params : 21.8 M
Start epoch  : 1


/home/vivek/.local/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:224: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


  [Ep   1/250] 100/415 loss=0.0566 ch=0.0000 gap=0.9913 ax=0.7027 ramp=0.00 lr=1.95e-06
  [Ep   1/250] 200/415 loss=0.0278 ch=0.0000 gap=0.9911 ax=0.7276 ramp=0.00 lr=3.87e-06
  [Ep   1/250] 300/415 loss=0.0224 ch=0.0000 gap=0.9903 ax=0.7149 ramp=0.00 lr=5.80e-06
  [Ep   1/250] 400/415 loss=0.0169 ch=0.0000 gap=0.9896 ax=0.6961 ramp=0.00 lr=7.73e-06
  [Ep   1/250] 415/415 loss=0.0161 ch=0.0000 gap=0.9895 ax=0.6265 ramp=0.00 lr=8.02e-06
[Epoch   1/250] train=2.1544 val=436.4255
  Chamfer=11.923±0.672 mm F@1=0.012 F@2=0.078 F@5=0.289 HD95=44.923
  ✓ New best val_loss: 436.4255
  ✓ New best Chamfer: 11.923 mm
  [Ep   2/250] 100/415 loss=0.0161 ch=0.0000 gap=0.9878 ax=0.6231 ramp=0.00 lr=9.95e-06
  [Ep   2/250] 200/415 loss=0.0170 ch=0.0000 gap=0.9852 ax=0.5405 ramp=0.00 lr=1.19e-05
  [Ep   2/250] 300/415 loss=0.0131 ch=0.0000 gap=0.9846 ax=0.5835 ramp=0.00 lr=1.38e-05
  [Ep   2/250] 400/415 loss=0.0127 ch=0.0000 gap=0.9825 ax=0.5954 ramp=0.00 lr=1.57e-05
  [Ep   2/250] 415/415 loss=0.0116

  [Ep  15/250] 100/415 loss=0.0129 ch=0.0000 gap=0.9360 ax=0.2687 ramp=0.00 lr=7.99e-05
  [Ep  15/250] 200/415 loss=0.0134 ch=0.0000 gap=0.9387 ax=0.3271 ramp=0.00 lr=7.99e-05
  [Ep  15/250] 300/415 loss=0.0135 ch=0.0000 gap=0.9472 ax=0.2419 ramp=0.00 lr=7.99e-05
  [Ep  15/250] 400/415 loss=0.0116 ch=0.0000 gap=0.9317 ax=0.2561 ramp=0.00 lr=7.99e-05
  [Ep  15/250] 415/415 loss=0.0116 ch=0.0000 gap=0.9354 ax=0.2819 ramp=0.00 lr=7.99e-05
[Epoch  15/250] train=0.3121 val=27.8326
  Chamfer=2.778±0.334 mm F@1=0.068 F@2=0.385 F@5=0.899 HD95=7.030
  [Ep  16/250] 100/415 loss=0.0132 ch=0.0000 gap=0.9479 ax=0.2037 ramp=0.00 lr=7.99e-05
  [Ep  16/250] 200/415 loss=0.0123 ch=0.0000 gap=0.9304 ax=0.2374 ramp=0.00 lr=7.99e-05
  [Ep  16/250] 300/415 loss=0.0154 ch=0.0000 gap=0.9463 ax=0.3082 ramp=0.00 lr=7.99e-05
  [Ep  16/250] 400/415 loss=0.0176 ch=0.0000 gap=0.9467 ax=0.2471 ramp=0.00 lr=7.99e-05
  [Ep  16/250] 415/415 loss=0.0102 ch=0.0000 gap=0.9309 ax=0.1874 ramp=0.00 lr=7.99e-05
[Epoch  16/25

  [Ep  29/250] 300/415 loss=0.2820 ch=0.0000 gap=0.9294 ax=0.0882 ramp=0.52 lr=7.88e-05
  [Ep  29/250] 400/415 loss=0.2807 ch=0.0000 gap=0.9371 ax=0.0932 ramp=0.52 lr=7.88e-05
  [Ep  29/250] 415/415 loss=0.3066 ch=0.0000 gap=0.9254 ax=0.2014 ramp=0.52 lr=7.88e-05
[Epoch  29/250] train=0.4440 val=21.4984
  Chamfer=2.415±0.453 mm F@1=0.095 F@2=0.481 F@5=0.940 HD95=5.566
  [Ep  30/250] 100/415 loss=0.2962 ch=0.0000 gap=0.9071 ax=0.1207 ramp=0.56 lr=7.87e-05
  [Ep  30/250] 200/415 loss=0.2867 ch=0.0000 gap=0.9162 ax=0.0745 ramp=0.56 lr=7.87e-05
  [Ep  30/250] 300/415 loss=0.2952 ch=0.0000 gap=0.9236 ax=0.0944 ramp=0.56 lr=7.87e-05
  [Ep  30/250] 400/415 loss=0.2822 ch=0.0000 gap=0.9023 ax=0.0773 ramp=0.56 lr=7.86e-05
  [Ep  30/250] 415/415 loss=0.2819 ch=0.0000 gap=0.8995 ax=0.0738 ramp=0.56 lr=7.86e-05
[Epoch  30/250] train=0.4544 val=19.5530
  Chamfer=2.324±0.468 mm F@1=0.099 F@2=0.504 F@5=0.951 HD95=5.202
  ✓ New best Chamfer: 2.324 mm
  [Ep  31/250] 100/415 loss=0.3070 ch=0.0000 gap=0.

...

  [Ep 181/250] 100/415 loss=0.3839 ch=0.0000 gap=0.7282 ax=0.0136 ramp=1.00 lr=1.55e-05
  [Ep 181/250] 200/415 loss=0.4175 ch=0.0000 gap=0.7973 ax=0.0145 ramp=1.00 lr=1.54e-05
  [Ep 181/250] 300/415 loss=0.4301 ch=0.0000 gap=0.8144 ax=0.0201 ramp=1.00 lr=1.54e-05
  [Ep 181/250] 400/415 loss=0.3959 ch=0.0000 gap=0.7508 ax=0.0154 ramp=1.00 lr=1.53e-05
  [Ep 181/250] 415/415 loss=6.3909 ch=5.9404 gap=0.8395 ax=0.0502 ramp=1.00 lr=1.52e-05
[Epoch 181/250] train=0.4893 val=30.1136
  Chamfer=2.514±0.438 mm F@1=0.111 F@2=0.461 F@5=0.915 HD95=8.039
  [Ep 182/250] 100/415 loss=0.4110 ch=0.0000 gap=0.7779 ax=0.0187 ramp=1.00 lr=1.51e-05
  [Ep 182/250] 200/415 loss=0.3830 ch=0.0000 gap=0.7239 ax=0.0146 ramp=1.00 lr=1.50e-05
  [Ep 182/250] 300/415 loss=0.4172 ch=0.0000 gap=0.7912 ax=0.0185 ramp=1.00 lr=1.49e-05
  [Ep 182/250] 400/415 loss=0.4014 ch=0.0000 gap=0.7613 ax=0.0159 ramp=1.00 lr=1.48e-05
  [Ep 182/250] 415/415 loss=0.4022 ch=0.0000 gap=0.7672 ax=0.0124 ramp=1.00 lr=1.48e-05
[Epoch 182/25

  [Ep 196/250] 100/415 loss=0.4656 ch=0.0000 gap=0.8676 ax=0.0167 ramp=1.00 lr=9.84e-06
  [Ep 196/250] 200/415 loss=0.3975 ch=0.0000 gap=0.7532 ax=0.0150 ramp=1.00 lr=9.76e-06
  [Ep 196/250] 300/415 loss=0.4273 ch=0.0000 gap=0.8123 ax=0.0142 ramp=1.00 lr=9.68e-06
  [Ep 196/250] 400/415 loss=0.4217 ch=0.0000 gap=0.8028 ax=0.0176 ramp=1.00 lr=9.60e-06
  [Ep 196/250] 415/415 loss=0.4076 ch=0.0000 gap=0.7755 ax=0.0150 ramp=1.00 lr=9.58e-06
[Epoch 196/250] train=0.4730 val=30.1450
  Chamfer=2.517±0.438 mm F@1=0.110 F@2=0.460 F@5=0.916 HD95=8.058
  [Ep 197/250] 100/415 loss=0.3984 ch=0.0000 gap=0.7555 ax=0.0149 ramp=1.00 lr=9.50e-06
  [Ep 197/250] 200/415 loss=0.4155 ch=0.0000 gap=0.7836 ax=0.0160 ramp=1.00 lr=9.42e-06
  [Ep 197/250] 300/415 loss=1.1619 ch=0.7448 gap=0.7887 ax=0.0263 ramp=1.00 lr=9.34e-06
  [Ep 197/250] 400/415 loss=0.4043 ch=0.0000 gap=0.7690 ax=0.0126 ramp=1.00 lr=9.26e-06
  [Ep 197/250] 415/415 loss=0.4021 ch=0.0000 gap=0.7664 ax=0.0141 ramp=1.00 lr=9.25e-06
[Epoch 197/25

  [Ep 211/250] 100/415 loss=0.3878 ch=0.0000 gap=0.7394 ax=0.0110 ramp=1.00 lr=5.30e-06
  [Ep 211/250] 200/415 loss=0.4000 ch=0.0000 gap=0.7644 ax=0.0096 ramp=1.00 lr=5.23e-06
  [Ep 211/250] 300/415 loss=0.4223 ch=0.0000 gap=0.8071 ax=0.0122 ramp=1.00 lr=5.17e-06
  [Ep 211/250] 400/415 loss=0.4072 ch=0.0000 gap=0.7749 ax=0.0130 ramp=1.00 lr=5.11e-06
  [Ep 211/250] 415/415 loss=0.4660 ch=0.0000 gap=0.8904 ax=0.0139 ramp=1.00 lr=5.10e-06
[Epoch 211/250] train=0.4677 val=30.2928
  Chamfer=2.525±0.438 mm F@1=0.109 F@2=0.457 F@5=0.915 HD95=8.054
  [Ep 212/250] 100/415 loss=0.4049 ch=0.0000 gap=0.7732 ax=0.0120 ramp=1.00 lr=5.04e-06
  [Ep 212/250] 200/415 loss=0.3268 ch=0.0000 gap=0.6117 ax=0.0126 ramp=1.00 lr=4.98e-06
  [Ep 212/250] 300/415 loss=1.2231 ch=0.8278 gap=0.7480 ax=0.0233 ramp=1.00 lr=4.92e-06
  [Ep 212/250] 400/415 loss=0.3980 ch=0.0000 gap=0.7609 ax=0.0105 ramp=1.00 lr=4.86e-06
  [Ep 212/250] 415/415 loss=0.3993 ch=0.0000 gap=0.7551 ax=0.0157 ramp=1.00 lr=4.85e-06
[Epoch 212/25

  [Ep 226/250] 100/415 loss=0.3969 ch=0.0000 gap=0.7555 ax=0.0122 ramp=1.00 lr=2.08e-06
  [Ep 226/250] 200/415 loss=0.4078 ch=0.0000 gap=0.7801 ax=0.0118 ramp=1.00 lr=2.04e-06
  [Ep 226/250] 300/415 loss=0.4090 ch=0.0000 gap=0.7757 ax=0.0115 ramp=1.00 lr=2.00e-06
  [Ep 226/250] 400/415 loss=0.3899 ch=0.0000 gap=0.7395 ax=0.0151 ramp=1.00 lr=1.96e-06
  [Ep 226/250] 415/415 loss=0.3947 ch=0.0000 gap=0.7511 ax=0.0141 ramp=1.00 lr=1.96e-06
[Epoch 226/250] train=0.4637 val=30.5261
  Chamfer=2.530±0.437 mm F@1=0.109 F@2=0.456 F@5=0.914 HD95=8.113
  [Ep 227/250] 100/415 loss=0.3899 ch=0.0000 gap=0.7431 ax=0.0103 ramp=1.00 lr=1.92e-06
  [Ep 227/250] 200/415 loss=0.4247 ch=0.0000 gap=0.8108 ax=0.0117 ramp=1.00 lr=1.88e-06
  [Ep 227/250] 300/415 loss=0.4074 ch=0.0000 gap=0.7763 ax=0.0115 ramp=1.00 lr=1.84e-06
  [Ep 227/250] 400/415 loss=0.3605 ch=0.0000 gap=0.6808 ax=0.0122 ramp=1.00 lr=1.80e-06
  [Ep 227/250] 415/415 loss=0.3776 ch=0.0000 gap=0.7151 ax=0.0128 ramp=1.00 lr=1.80e-06
[Epoch 227/25

  [Ep 241/250] 100/415 loss=0.4078 ch=0.0000 gap=0.7780 ax=0.0130 ramp=1.00 lr=3.26e-07
  [Ep 241/250] 200/415 loss=0.4491 ch=0.0000 gap=0.8181 ax=0.0113 ramp=1.00 lr=3.10e-07
  [Ep 241/250] 300/415 loss=0.3996 ch=0.0000 gap=0.7625 ax=0.0108 ramp=1.00 lr=2.95e-07
  [Ep 241/250] 400/415 loss=0.3847 ch=0.0000 gap=0.7319 ax=0.0110 ramp=1.00 lr=2.79e-07
  [Ep 241/250] 415/415 loss=0.4009 ch=0.0000 gap=0.7607 ax=0.0123 ramp=1.00 lr=2.77e-07
[Epoch 241/250] train=0.4621 val=30.4346
  Chamfer=2.529±0.438 mm F@1=0.109 F@2=0.456 F@5=0.914 HD95=8.081
  [Ep 242/250] 100/415 loss=0.6332 ch=0.2497 gap=0.6847 ax=0.0601 ramp=1.00 lr=2.63e-07
  [Ep 242/250] 200/415 loss=0.4594 ch=0.0000 gap=0.8448 ax=0.0135 ramp=1.00 lr=2.48e-07
  [Ep 242/250] 300/415 loss=0.3980 ch=0.0000 gap=0.7572 ax=0.0128 ramp=1.00 lr=2.35e-07
  [Ep 242/250] 400/415 loss=0.4046 ch=0.0000 gap=0.7658 ax=0.0158 ramp=1.00 lr=2.21e-07
  [Ep 242/250] 415/415 loss=0.4121 ch=0.0000 gap=0.7806 ax=0.0164 ramp=1.00 lr=2.19e-07
[Epoch 242/25

In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# TEST EVALUATION
# ══════════════════════════════════════════════════════════════════════════════

print('Loading best checkpoint for test evaluation …')
best_ch_path = CKPT_DIR / 'best_chamfer_checkpoint.pth'
ckpt_to_use  = best_ch_path if best_ch_path.exists() else BEST_CKPT_PATH

state = torch.load(ckpt_to_use, map_location=device, weights_only=False)
model_eval = PPCNetV8().to(device)
model_eval.load_state_dict(state['model'])
model_eval.eval()
print(f'  Loaded epoch {state["epoch"] + 1}')

test_ds     = LumbarDatasetV8(test_ids, augment=False)
test_loader = DataLoader(test_ds, batch_size=2, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True,
                         collate_fn=collate_keep_strings)

all_metrics = []
with torch.no_grad():
    for batch in test_loader:
        batch  = move_to_device(batch)
        output = model_eval(batch['drr_ap'], batch['drr_lp'],
                            batch['P_ap'], batch['P_lp'],
                            batch['center'], batch['scale'])
        pred = output['pred_world'].cpu().numpy()
        gt   = batch['gt_ppc_world'].cpu().numpy()
        pids = batch['patient_id']
        for b in range(pred.shape[0]):
            m = chamfer_metrics_np(pred[b], gt[b])
            m['patient_id'] = pids[b]
            all_metrics.append(m)
            save_vtk_points(pred[b], RESULTS_DIR / f'{pids[b]}_pred.vtk')

print(f'\n{"="*60}')
print(f'TEST SET RESULTS ({len(all_metrics)} patients)')
print(f'{"="*60}')
keys = ['chamfer_mm','fscore_1mm','fscore_2mm','fscore_5mm','hausdorff_95']
labels = ['Chamfer (mm)','F@1mm','F@2mm','F@5mm','HD95 (mm)']
for k, lbl in zip(keys, labels):
    vals = [m[k] for m in all_metrics]
    print(f'  {lbl:<16} mean={np.mean(vals):.3f} std={np.std(vals):.3f} '
          f'median={np.median(vals):.3f}')

import csv
csv_path = RESULTS_DIR / 'test_results.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['patient_id'] + keys)
    writer.writeheader()
    writer.writerows(all_metrics)
print(f'Results saved to {csv_path}')

Loading best checkpoint for test evaluation …
  Loaded epoch 79

TEST SET RESULTS (105 patients)
  Chamfer (mm)     mean=2.485 std=1.479 median=2.220
  F@1mm            mean=0.112 std=0.049 median=0.103
  F@2mm            mean=0.503 std=0.132 median=0.510
  F@5mm            mean=0.937 std=0.093 median=0.962
  HD95 (mm)        mean=6.738 std=7.255 median=5.337
Results saved to /apps/app/pandu/ppc_network_v8/results/test_results.csv


In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# INFERENCE FUNCTION
# ══════════════════════════════════════════════════════════════════════════════

def predict_ppc(drr_ap_path, drr_lp_path, p_ap_path, p_lp_path,
                output_path, checkpoint_path=None,
                center_mm=None, scale_mm=None):
    if checkpoint_path is None:
        best_ch = CKPT_DIR / 'best_chamfer_checkpoint.pth'
        checkpoint_path = best_ch if best_ch.exists() else BEST_CKPT_PATH
    state = torch.load(checkpoint_path, map_location=device, weights_only=False)
    net = PPCNetV8().to(device)
    net.load_state_dict(state['model'])
    net.eval()
    print(f'Loaded checkpoint (epoch {state["epoch"] + 1})')
    img_norm = transforms.Normalize(mean=[0.485], std=[0.229])
    def _load(path):
        img = load_drr_image(path)
        return img_norm(torch.from_numpy(img).unsqueeze(0).unsqueeze(0).float()).to(device)
    drr_ap_t = _load(drr_ap_path)
    drr_lp_t = _load(drr_lp_path)
    P_ap_t = torch.from_numpy(load_projection_matrix(p_ap_path)).unsqueeze(0).float().to(device)
    P_lp_t = torch.from_numpy(load_projection_matrix(p_lp_path)).unsqueeze(0).float().to(device)
    if center_mm is None: center_mm = [0.0, 0.0, 0.0]
    if scale_mm is None:  scale_mm = [80.0, 80.0, 130.0]
    center_t = torch.tensor([center_mm], dtype=torch.float32, device=device)
    scale_t  = torch.tensor([scale_mm], dtype=torch.float32, device=device)
    with torch.no_grad():
        out = net(drr_ap_t, drr_lp_t, P_ap_t, P_lp_t, center_t, scale_t)
        pred = out['pred_world'].squeeze(0).cpu().numpy()
    save_vtk_points(pred, output_path)
    print(f'Saved {len(pred)} points → {output_path}')
    return pred

print('predict_ppc() ready.')

predict_ppc() ready.
